Se nos presenta el siguiente problema, a partir de una base de datos de acceso público sobre muchos electrogardiogramas de diversos pacientes.

"Los cardiólogos y técnicos en electrocardiografía analizan registros de ECG para detectar anomalías en el ritmo cardíaco. El problema es que un ECG de 12 derivaciones genera una cantidad enorme de datos, y leer manualmente miles de estudios es lento y propenso al error humano. En este contexto, no detectar una arritmia grave (falso negativo) puede tener consecuencias fatales para el paciente, mientras que un falso positivo genera una derivación innecesaria al cardiólogo, lo cual es costoso pero manejable.

El dataset PTB-XL contiene señales de ECG de pacientes reales anotadas por médicos, con etiquetas de diagnóstico que van desde "normal" hasta distintos tipos de arritmias e infartos. Cada registro incluye además metadatos del paciente como edad, sexo e indicadores técnicos de la señal. El objetivo es clasificar cada estudio en su categoría diagnóstica correspondiente."

Luego de entrar al link de donde se puede descargar esta base de datos, observamos:
1) que es muy grande; es probable que se requiera dividir el conjunto de entrenamiento en minibatches para un entrenamiento eficiente.
2) que hay 5 clases, de las cuales sólo una es "negativa" (ECG normal). Las otras 4 son patologías con diferentes grados de riesgo. En este trabajo no disponemos de profesionales de la medicina o la cardiología que nos asesoren. Luego de preguntarle a Claude por el significado clínico y nivel de riesgo de esas patologías, concluimos que las clases patológicas se dividen de la siguiente forma:
- MI (infarto de miocardio): riesgo altísimo, muchas veces es letal a corto plazo;
- STTC y CD: es importante detectarlos, pero muchas veces no suponen riesgos a corto plazo;
- HYP (hipertrofia): supone riesgos solamente a largo plazo. Se puede considerar como "el menos grave".

Se nos pide clasificar las instancias lo mejor posible entre las 5 clases y así trabajaremos, pero priorizando fuertemente que no se nos escapen infartos de miocardio dado que suelen ser letales rápidamente.

Debido a la naturaleza de los datos disponibles y su cantidad, que más abajo describiremos, se decide que la arquitectura adecuada será una red neuronal. Luego reflexionaremos sobre qué características debería tener según la naturaleza de los atributos, de manera de guiar una construcción eficaz.

In [1]:
import requests
import pandas as pd
import numpy as np
import ast
from pathlib import Path

In [2]:
base_url = "https://physionet.org/files/ptb-xl/1.0.3/"
local_dir = Path("./ptb-xl/")

# Descargar el CSV primero
csv_path = local_dir / "ptbxl_database.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
with open(csv_path, 'wb') as f:
    f.write(requests.get(base_url + "ptbxl_database.csv").content)
# Descargar las señales a 100 Hz
df = pd.read_csv(csv_path)

KeyboardInterrupt: 

In [3]:
local_dir = Path("./ptb-xl/")
csv_path = local_dir / "ptbxl_database.csv"
df = pd.read_csv(csv_path)

In [4]:
df = pd.read_csv("ptbxl_database.csv")
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

In [4]:
df['filename_lr'].head()

0    records100/00000/00001_lr
1    records100/00000/00002_lr
2    records100/00000/00003_lr
3    records100/00000/00004_lr
4    records100/00000/00005_lr
Name: filename_lr, dtype: object

In [13]:
RETOMAR_DESDE = "records100/05000/05762_lr"  # Cambiar según necesidad
todos_los_archivos = []
for path in df['filename_lr']:
    todos_los_archivos.append(path + '.hea')
    todos_los_archivos.append(path + '.dat')
if RETOMAR_DESDE+'.hea' in todos_los_archivos:
    idx_inicio = todos_los_archivos.index(RETOMAR_DESDE+'.hea')
else:
    raise ValueError(f"No se encontró '{RETOMAR_DESDE}' en la lista de archivos.")

# Descargar desde ese punto
for path in todos_los_archivos[idx_inicio:]:
    url = base_url + path + ''
    local_file = local_dir / (path + '')
    local_file.parent.mkdir(parents=True, exist_ok=True)
    if not local_file.exists():
        with open(local_file, 'wb') as f:
            f.write(requests.get(url).content)
        #print(f"Descargado: {path + ''}")

In [5]:
local_dir = Path("./ptb-xl/")

# Cargar metadata
df = pd.read_csv(local_dir / "ptbxl_database.csv", index_col="ecg_id")

# Convertir scp_codes de string a diccionario
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

EmptyDataError: No columns to parse from file

In [6]:
df #21.799 casos totales (aunque el último ecg_id es 21.837)



,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,True,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,True,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,True,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,True,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr


In [7]:
df['strat_fold'].unique()

array([ 3,  2,  5,  4,  7,  9, 10,  8,  6,  1], dtype=int64)

In [8]:
superclass_map = {
    'NORM': 'NORM', 
    'MI': 'MI', 'AMI': 'MI', 'IMI': 'MI', 'ASMI': 'MI', 'ILMI': 'MI',
    'INJAS': 'MI', 'INJAL': 'MI', 'IPLMI': 'MI', 'IPMI': 'MI',
    'INJIN': 'MI', 'INJLA': 'MI', 'LMI': 'MI', 'PMI': 'MI', 'INJIL': 'MI',
    'STTC': 'STTC', 'NST_': 'STTC', 'DIG': 'STTC', 'LNGQT': 'STTC',
    'ANEUR': 'STTC', 'EL': 'STTC', 'ISCI': 'STTC', 'ISC_': 'STTC',
    'ISCA': 'STTC',
    'CD': 'CD', 'LAFB': 'CD', 'IRBBB': 'CD', '1AVB': 'CD', 'IVCD': 'CD',
    'CRBBB': 'CD', 'CLBBB': 'CD', 'ILBBB': 'CD', 'WPW': 'CD',
    'LPFB': 'CD', '2AVB': 'CD', '3AVB': 'CD', 'LFBAS': 'CD',
    'LAFB/LPFB': 'CD',
    'HYP': 'HYP', 'LVH': 'HYP', 'RVH': 'HYP', 'SEHYP': 'HYP',
    'LVQRSSL': 'HYP', 'LVQRSLL': 'HYP', 'LAO/LAE': 'HYP', 'RAO/RAE': 'HYP',
}

def get_superclasses(scp_codes):
    superclasses = set()
    for code, likelihood in scp_codes.items():
        if likelihood >= 50 and code in superclass_map:
            superclasses.add(superclass_map[code])
    return list(superclasses) if superclasses else None

df['superclasses'] = df['scp_codes'].apply(get_superclasses)

# Filtrar registros sin etiqueta asignable

In [9]:
df_diagnosticados = df[df['superclasses'].notna()] #hay 18461 casos con esta característica (sólo podemos usar dentro de estos)

La página web de los creadores de la base de datos https://physionet.org/content/ptb-xl/1.0.3/#files-panel nos informa cuántos casos hay para cada uno de los 5 diagnósticos. Son 9514 para NORM (ECG normal), 5469 para MI (infarto de miocardio), 5235 para STTC, 4898 para CD y 2649 para HYP. Hay desbalance de clases, por lo que debemos tener cuidado con las métricas a utilizar y con cómo separamos los datos. Luego de apartar las instancias de desarrollo de las de evaluación, tendremos que elegir bien cómo dividir los de desarrollo. Concretamente, hará falta un Stratified K-Fold para mantener una presencia equilibrada de cada clase en cada fold. Por otro lado, debemos recordar el contexto del problema. Cada estudio corresponde a un paciente, y para algunos de ellos hay más de un estudio como veremos más abajo. Luego, tendremos que hacer un Group K-Fold estratificado, para que no haya estudios de un paciente en diferentes folds. Esto se debe a que queremos usar la información de pacientes que ya tuvimos para el futuro, es decir, para predecir diagnósticos de pacientes nuevos. Por eso, tampoco podemos permitir que un mismo paciente aparezca en los conjuntos de desarrollo y de test.

La base de datos ya resuelve esto: tiene un atributo llamado strat_fold, con enteros que van del 1 al 10. Esto corresponde a cómo les quedó a los creadores su Stratified Group K-Fold con K=10, que respeta todos los criterios de los que hablamos antes. Usaremos estos folds, y también seguiremos su recomendación de usar los primeros 8 para training, el 9 para validación y el 10 para test.

Por otro lado, como pueden convivir varios diagnósticos en un mismo estudio, predecir los correctos es un problema de clasificación multietiqueta.

In [10]:
df_diagnosticados.columns

Index(['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse',
       'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis',
       'infarction_stadium1', 'infarction_stadium2', 'validated_by',
       'second_opinion', 'initial_autogenerated_report', 'validated_by_human',
       'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems',
       'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr',
       'superclasses'],
      dtype='object')

In [11]:
df_diagnosticados[df_diagnosticados['height'].notna()] #Sólo 6002 casos tienen registro de altura
df_diagnosticados[df_diagnosticados['weight'].notna()] #Sólo 8297 casos tienen registro de peso
df_diagnosticados[df_diagnosticados['age'].notna()] #todos los casos tienen edad
df_diagnosticados[df_diagnosticados['patient_id'].notna()] #todos tienen id de paciente
df_diagnosticados[df_diagnosticados['static_noise'].notna()] #2735 casos tienen static_noise no nulo
df_diagnosticados[df_diagnosticados['burst_noise'].notna()] #488 casos
df_diagnosticados[df_diagnosticados['electrodes_problems'].notna()] #27 casos
df_diagnosticados[df_diagnosticados['extra_beats'].notna()] #1564 casos
df_diagnosticados[df_diagnosticados['report'].notna()] #todos tienen
df_diagnosticados['report'].head(30) #descripciones de lo visto en el estudio
df_diagnosticados[df_diagnosticados['baseline_drift'].notna()] #1376 casos
df_diagnosticados[df_diagnosticados['initial_autogenerated_report'].notna()] #todos tienen
df_diagnosticados['initial_autogenerated_report'].head(30) #son booleanos
df_diagnosticados[df_diagnosticados['infarction_stadium1'].notna()] #4738 casos, son strings, muchos desconocidos
df_diagnosticados[df_diagnosticados['infarction_stadium2'].notna()][['infarction_stadium2','superclasses']] #100 casos no nulos, probablemente todos con infarto
df_diagnosticados[df_diagnosticados['nurse'].notna()] #17.182 casos, son variables numéricas (float)
df_diagnosticados[df_diagnosticados['site'].notna()] #casi todos (18.446 casos)
df_diagnosticados[df_diagnosticados['recording_date'].notna()] #todos tienen
df_diagnosticados[df_diagnosticados['device'].notna()] #todos tienen, son strings que combinan letras y números, existen 11 diferentes
df_diagnosticados[df_diagnosticados['heart_axis'].notna()]['heart_axis'].head(30) #11.378 casos, son strings
df_diagnosticados[df_diagnosticados['validated_by'].notna()]['validated_by'].head() #10.645 casos, son floats
df_diagnosticados[df_diagnosticados['validated_by_human'].notna()]['validated_by_human'].unique() #todos, son booleanos, hay verdaderos y falsos
df_diagnosticados[df_diagnosticados['second_opinion'].notna()]['second_opinion'].head(30) #todos, son booleanos
df_diagnosticados[df_diagnosticados['sex'].notna()]['sex'].unique() #todos, son ceros y unos (enteros)
len(df_diagnosticados['patient_id'].unique()) #hay en total 16.347 pacientes distintos
df_diagnosticados[df_diagnosticados['pacemaker'].notna()] #12 casos

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,superclasses
1612,1625,5233.0,300.0,1,148.0,32.0,10.0,1.0,AT-6 6,1988-04-29 09:42:54,...,"v1,",NaN,NaN,NaN,NaN,"PACE????, nan",8,records100/01000/01625_lr,records500/01000/01625_hr,[CD]
1818,1831,18924.0,79.0,1,NaN,NaN,1.0,2.0,CS-12,1988-07-02 15:03:10,...,NaN,NaN,NaN,NaN,SVES,"ja, nan",7,records100/01000/01831_lr,records500/01000/01831_hr,[NORM]
2171,2184,2120.0,53.0,1,167.0,58.0,5.0,1.0,AT-6 C 5.5,1988-09-05 14:48:09,...,NaN,NaN,NaN,NaN,NaN,"?, nan",6,records100/02000/02184_lr,records500/02000/02184_hr,[MI]
4077,4097,648.0,40.0,1,155.0,51.0,5.0,1.0,AT-6 C 5.5,1990-04-30 11:07:33,...,NaN,NaN,NaN,NaN,1ES,"PACE????, nan",10,records100/04000/04097_lr,records500/04000/04097_hr,[NORM]
7989,8013,4299.0,67.0,0,168.0,69.0,1.0,1.0,AT-6 C 5.8,1992-04-23 11:58:37,...,NaN,NaN,NaN,NaN,NaN,"ja, pacemaker",7,records100/08000/08013_lr,records500/08000/08013_hr,"[MI, STTC]"
8133,8157,15179.0,300.0,1,NaN,NaN,1.0,2.0,CS-12,1992-05-23 13:12:42,...,NaN,NaN,NaN,NaN,NaN,"ja, pacemaker",5,records100/08000/08157_lr,records500/08000/08157_hr,[MI]
9550,9574,16131.0,47.0,1,NaN,NaN,1.0,2.0,CS-12,1993-01-03 17:49:35,...,NaN,", I-AVR,",NaN,NaN,NaN,"ja, nan",2,records100/09000/09574_lr,records500/09000/09574_hr,[NORM]
10491,10518,21159.0,68.0,0,NaN,NaN,1.0,2.0,CS-12,1993-07-14 09:31:08,...,NaN,NaN,NaN,NaN,NaN,"ja, pacemaker",9,records100/10000/10518_lr,records500/10000/10518_hr,"[CD, HYP]"
10819,10846,10796.0,83.0,0,NaN,NaN,1.0,2.0,CS-12,1993-08-17 11:58:25,...,", V2",NaN,NaN,NaN,VES,"ja, pacemaker",10,records100/10000/10846_lr,records500/10000/10846_hr,[NORM]
13215,13247,10420.0,54.0,0,NaN,NaN,0.0,0.0,CS100 3,1994-10-26 08:40:49,...,NaN,", I-AVF,",NaN,NaN,NaN,"ja, pacemaker",6,records100/13000/13247_lr,records500/13000/13247_hr,"[STTC, HYP]"


Hemos explorado todas las columnas del dataframe de metadata de los electrocardiogramas, primero para ver cuántos casos había donde cada atributo no fuera Null. Antes de hacer esto, eliminamos los casos sin superclases, por no tener un diagnóstico que podamos predecir o aprender. En los casos donde no era obvio, hemos visto también el tipo de dato. Con esta información, sumada al contexto, decidimos qué información tendremos en cuenta para construir nuestra red neuronal; damos los motivos en el caso de los atributos descartados.

Esta base de datos tiene atributos filename_lr y filename_hr, correspondientes a las carpetas records100 y record500 respectivamente. Las dos tienen registros de los mismos electrocardiogramas, pero records100 almacena 1000 instantes de tiempo y records500 almacena 5000. Como la última pesa 2.5GB y este estudio es para un portfolio, usamos la primera para no excedernos en memoria y tiempo. Luego, tomaremos el atributo filename_hr porque nos permitirá conectar los datos del paciente con su electrocardiograma (de records100), y descartaremos filename_hr (que tiene la versión de records500). 

Usaremos la edad y el sexo de los pacientes, dado que podrían ser útiles para el diagnóstico. La altura y el peso serían de mucha utilidad (sobre todo la relación entre ambos), pero hay muy pocos casos que no sean nulos. Es decir, en la gran mayoría de los casos no tenemos esta información. Más precisamente, tanto altura como peso tienen menos de 10.000 registros no nulos, y hemos visto que hay más de 16.000 pacientes, por lo que necesariamente no se puede recuperar la altura y peso de muchos de ellos. No queremos inventarlas porque esto puede afectar demasiado a los resultados (dado que la relación peso-altura habla mucho de la salud de una persona). Descartamos initial_autogenerated_report, nurse, site y recording_date porque no los creemos útiles para el diagnóstico. Infaction_stadium 1 y 2 tienen muchos nulls, y además están asociados a detección de infartos (sobre todo el 2). Nosotros queremos predecir infartos directamente desde las mediciones de los electrocardiogramas, para poder prevenir estas eventualidades. Luego, tampoco usaremos los infarction_stadium. Por el mismo motivo descartamos los report, que son descripciones en palabras de los diagnósticos. La página web de los creadores de la base de datos nos informa que todos los registros fueron validados por expertos, así que despreciamos los atributos validated_by y validated_by_human. Usaremos el patient_id porque identifica al paciente, ecg_id como clave de las instancias y el atributo generado superclasses que agrega todos los diagnósticos en 5 grupos (las clases que queremos aprender a predecir). Luego, no usamos el atributo scp_codes que luego fue agregado en superclasses. Tomaremos todos los registros de las 12 derivaciones de los electrocardiogramas. Sobre el atributo device, aún no sabemos si es importante o no. Podemos intentar decidirlo haciendo estudios estadísticos o decidir no tenerlo en cuenta (sabiendo que tal vez era necesario) y luego analizar los resultados. Todos los atributos no mencionados hasta ahora se descartan por tener demasiados Nulls. 

En este estudio, dado que no es para un hospital real sino para un portfolio, decidimos no incluir el atributo device a fin de reducir la complejidad del proyecto.

Por lo tanto, definimos el nuevo dataframe con los atributos que usaremos:

In [12]:
df_final = df_diagnosticados[['patient_id','sex','age','strat_fold','filename_lr','superclasses']]

In [19]:
df_final

,patient_id,sex,age,strat_fold,filename_lr,superclasses
0,15709.0,1,56.0,3,records100/00000/00001_lr,[NORM]
1,13243.0,0,19.0,2,records100/00000/00002_lr,[NORM]
2,20372.0,1,37.0,5,records100/00000/00003_lr,[NORM]
3,17014.0,0,24.0,3,records100/00000/00004_lr,[NORM]
4,17448.0,1,19.0,4,records100/00000/00005_lr,[NORM]
...,...,...,...,...,...,...
21792,11905.0,1,55.0,9,records100/21000/21831_lr,[NORM]
21793,7954.0,0,63.0,7,records100/21000/21832_lr,[CD]
21795,20703.0,0,300.0,4,records100/21000/21834_lr,[NORM]
21797,8873.0,1,64.0,8,records100/21000/21836_lr,[NORM]


Como aparece la edad 300, que no puede ser la edad real de una persona, averiguamos que el número 300 se usa para mayores de 89 años, así que reemplazamos el 300 por 90.

Habiendo completado nuestro estudio preliminar, es momento de pasar toda la información de atributos y clases de las instancias al formato que usaremos para conectarlo con nuestra red neuronal: tensores de Pytorch. Luego diseñaremos la arquitectura de la red.

Como la web nos informa que los ecg_id son únicos, hay 18.461 casos y el último id es 21.837, vemos que faltan algunos. Estos id se corresponden con el nombre del archivo que contiene el electrocardiograma, pero ya nos son necesarios. Nos basta saber que cada fila corresponde a un electrocardiograma diferente. Por lo tanto, reindexaremos la tabla para que no haya confusiones con los índices de los tensores.

In [16]:
df_indexado = df_final.reset_index(drop=True)
df_indexado.loc[df_indexado['age'] == 300, 'age'] = 90

In [ ]:
df_indexado

,patient_id,sex,age,strat_fold,filename_lr,superclasses
0,15709.0,1,56.0,3,records100/00000/00001_lr,[NORM]
1,13243.0,0,19.0,2,records100/00000/00002_lr,[NORM]
2,20372.0,1,37.0,5,records100/00000/00003_lr,[NORM]
3,17014.0,0,24.0,3,records100/00000/00004_lr,[NORM]
4,17448.0,1,19.0,4,records100/00000/00005_lr,[NORM]
...,...,...,...,...,...,...
18456,11905.0,1,55.0,9,records100/21000/21831_lr,[NORM]
18457,7954.0,0,63.0,7,records100/21000/21832_lr,[CD]
18458,20703.0,0,90.0,4,records100/21000/21834_lr,[NORM]
18459,8873.0,1,64.0,8,records100/21000/21836_lr,[NORM]


In [18]:
df_indexado.to_csv("ptbxl_reduced.csv")

A continuación comenzamos a usar Pytorch para pasar la matriz de targets y los valores de los electrocardiogramas a tensores.